#Thuật toán A*

In [ ]:
from collections import deque

class Graph:
  def __init__(self, adjacency_list):
    self.adjacency_list = adjacency_list

  def get_neighbors(self, v): #Lấy các node cận kề node hiện tại
    return self.adjacency_list[v]

  def h(self, n): #Hàm tính heuristic
    H = {
      'A': 1,
      'B': 1,
      'C': 1,
      'D': 1
    }

    return H[n]

  def a_star_algorithm(self, start_node, stop_node): #Thuật toán A*, truyền vào 2 tham số node bắt đầu và node đích
    open_list = set([start_node]) #Bước 1: Xét node bắt dầu
    closed_list = set([]) #Bước 1: closed_list rỗng

    poo = {}
    poo[start_node] = 0 #g[n]

    par = {}
    par[start_node] = start_node #lưu dấu vết

    while len(open_list) > 0: #Chạy cho đến khi không còn node nào để xét nữa
      n = None

      for v in open_list:
        if n == None or poo[v] + self.h(v) < poo[n] + self.h(n): #tìm node có chi phí f[n] min
          n = v         #sau khi tìm gán n = node có chi phí f[n] min

      if n == None:
        print('Path does not exist!')
        return None

      if n == stop_node: #sau khi tìm thấy đích
        reconst_path = []

        while par[n] != n:
          reconst_path.append(n)
          n = par[n] #truy vết

        reconst_path.append(start_node)

        reconst_path.reverse() #đảo ngược lại từ đích tới đầu

        print('Path found: {}'.format(reconst_path))
        return reconst_path #trả lại kết quả

      for(m, weight) in self.get_neighbors(n): #trọng số các node cận kề
        if m not in open_list and m not in closed_list: #nếu m chưa được xét
          open_list.add(m) #thêm m vào open_list để xét
          par[m] = n #truy vết
          poo[m] = poo[n] + weight #g[m] = g[n] + trọng số
        else:
          if poo[m] > poo[n] + weight:
            poo[m] = poo[n] + weight
            par[m] = n #lấy chi phí min

            if m in closed_list:
              closed_list.remove(m)
              open_list.add(m)
      open_list.remove(n)
      closed_list.add(n)
    print('Path does not exist!')
    return None

adjacancy_list = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('D', 5)],
    'C': [('D', 12)]
}

graph1= Graph(adjacancy_list)
graph1.a_star_algorithm('A', 'D') #Start_node: A, Stop_node: D

Path found: ['A', 'B', 'D']


['A', 'B', 'D']

#Thuật toán AKT

In [1]:
import copy
from heapq import heappush, heappop

#Giả sử 8-puzzle (n=3)
n = 3

rows = [1,0,-1,0]
cols = [0,-1,0,1]

class priorityQueue:
  def __init__(self):
    self.heap = []

  def push(self, key):
    heappush(self.heap, key)

  def pop(self):
    return heappop(self.heap)

  def empty(self):
    if not self.heap:
      return True
    else:
      return False

class nodes:
  def __init__(self, parent, mats, empty_tile_pos, costs, levels):
    self.parent = parent
    self.mats = mats
    self.empty_tile_pos = empty_tile_pos
    self.costs = costs
    self.levels = levels

  def __lt__(self, nxt):
    return self.costs < nxt.costs

#Tính toán chi phí (tức là tính toán những ô sai so với kết quả muốn trả về)
  def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
      for j in range(n):
        # We only count misplaced tiles that are not the empty tile (0)
        if mats[i][j] != 0 and mats[i][j] != final[i][j]:
          count += 1
    return count

  def newNodes(mats, empty_tile_posi, new_empty_tile_posi, level, parent, final):
    new_mats = copy.deepcopy(mats)
    x1 = empty_tile_posi[0]
    x2 = new_empty_tile_posi[0]
    y1 = empty_tile_posi[1]
    y2 = new_empty_tile_posi[1]

    # Swap tiles
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Calculate costs for the new state
    costs = nodes.calculateCosts(new_mats, final)

    # Create and return a new node instance
    new_node = nodes(parent, new_mats, new_empty_tile_posi, costs, level)
    return new_node


  def printMatrix(mats):
    for i in range(n):
      for j in range(n):
        print("%d " % (mats[i][j]), end = " ")
      print()


  def isSafe(x, y):
    return x >=0 and x<n and y >= 0 and y<n


  def printPath(root):
    if root==None:
      return
    nodes.printPath(root.parent)
    nodes.printMatrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
  pq = priorityQueue()

  # Initial cost calculation for the root node
  costs = nodes.calculateCosts(initial, final) # Corrected call: removed empty_tile_posi
  root = nodes(None, initial, empty_tile_posi, costs, 0)

  pq.push(root)
  while not pq.empty():
    minimum = pq.pop()

    # Check if goal state is reached (cost is 0)
    if minimum.costs == 0:
      nodes.printPath(minimum)
      return

    # Explore neighbors
    for i in range(len(rows)): # Iterate through possible moves
      new_tile_posi=[
          minimum.empty_tile_pos[0] + rows[i],
          minimum.empty_tile_pos[1] + cols[i],
      ]

      # If the new position is safe, create a child node
      if nodes.isSafe(new_tile_posi[0], new_tile_posi[1]):
        child = nodes.newNodes(
            minimum.mats,
            minimum.empty_tile_pos,
            new_tile_posi,
            minimum.levels + 1, # Use levels from minimum node
            minimum,
            final
        )
        pq.push(child)

initial = [[1,2,3], [5,6,0], [7,8,4]] #Ban đầu
empty_tile_posi = [1,2]
final = [[1, 2, 3], [5, 8, 6], [0, 7, 4]] #Kết quả muốn trả về
solve(initial, empty_tile_posi, final)

1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  

